In [109]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from dataclasses import dataclass

load_dotenv()

True

In [110]:
@dataclass(frozen=True)
class Provider:
    """Provider class to return the available provider based on the environment variable."""

    name: str
    env_var: str
    base_url: str
    model: str

In [111]:
PROVIDERS = [
    Provider(
        name="Open Router",
        env_var="OPEN_ROUTER_API_KEY",
        base_url="https://openrouter.ai/api/v1",
        model="openai/gpt-4o"
    ),
    Provider(
        name="Gemini",
        env_var="GEMINI_API_KEY",
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        model="gemini-3.5-flash"
    ),
    Provider(
        name="Groq",
        env_var="GROQ_API_KEY",
        base_url="https://api.groq.com/openai/v1",
        model="openai/gpt-oss-20b"
    )
]

In [112]:
import random

def select_provider() -> Provider:
    """Randomly select one of the configured providers."""

    available_providers = [
        provider
        for provider in PROVIDERS
        if os.getenv(provider.env_var)
    ]

    if not available_providers:
        raise ValueError(
            "No valid provider found. Please set the appropriate environment variable."
        )

    return random.choice(available_providers)

In [113]:
def build_client(provider: Provider) -> OpenAI:
    """Build the OpenAI client based on the selected provider."""
    api_key = os.getenv(provider.env_var)
    if not api_key:
        raise ValueError(f"API key for {provider.name} is not set in environment variables.")
    
    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url
    )

In [114]:
provider = select_provider()
client = build_client(provider)

In [115]:
SYSTEM_PROMPT = """
    You are a mathematics assistant.
    You will receive a maths problem in simple string.
    You have to solve that problem with chain of thought problem
    You have to solve in step by step way
    After the complete solution you have to return answer in below format:-
        Final Answer: <numerical_answer>
    After Final Answer: <numerical_answer> dont add any text or anything
"""

In [116]:
def llm_reply(prompt, temperature: float =1):
    response = client.chat.completions.create(
        model=provider.model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content

In [117]:
def extract_final_number(answer: str | None) -> float:
    if not answer or not answer.split():
        raise ValueError("The response does not contain a final answer.")

    number_text = answer.split()[-1]
    number = float(number_text)
    return number

In [118]:
PROMPT = """
    How many distinct ways can 8 identical balls be distributed among 4 distinct
    boxes if each box must contain at least 1 ball and no box can contain more
    than 4 balls?
"""

In [119]:
raw_response = llm_reply(PROMPT)
numeric_answer = extract_final_number(raw_response)

In [120]:
pass1 = "She has 16 - 3 - 4 = 9 eggs left. So she makes $2 * 9 = $18 per day. Final Answer: 18"
pass2 = "She eats 3 for breakfast, so she has 16 - 3 = 13 left. Then she bakes muffins, so she has 13 - 4 = 9 eggs left. So she has 9 eggs * $2 = $18. Final Answer: 18"
pass3 = "She uses 3 eggs for breakfast and 4 for muffins, leaving 16 - 7 = 9 eggs. At $2 each, she earns 9 * 2 = $18. Final Answer: 18"
fail1 = "This means she sells the remainder for $2 * (16 - 4 - 3) = $26 per day. Final Answer: 26"
fail2 = "She sells all 16 eggs at $2 each before accounting for the eggs she uses, so she makes 16 * 2 = $32 per day. Final Answer: 32"

In [121]:
print(extract_final_number(fail1))

26.0


In [122]:
reasoning_samples = [pass1, pass2, pass3, fail1, fail2]

In [123]:
@dataclass
class VoteResult:
    winner: float | None
    tally: dict
    samples: int
    confidence: float

In [124]:
def majority_vote(answers):
    tally = {}

    for answer in answers:
        number = extract_final_number(answer)

        if number not in tally:
            tally[number]=0
            
        tally[number]+=1

    winner = None
    max_votes = 0

    for number,votes in tally.items():
        if votes>max_votes:
            max_votes = votes
            winner = number

    return winner,tally

In [125]:
def self_consistent_answer(reasoning_samples):
    winner, tally = majority_vote(reasoning_samples)

    samples = len(reasoning_samples)
    confidence = tally[winner]/samples

    return VoteResult(
        winner=winner,
        tally=tally,
        samples=samples,
        confidence=confidence
    )

In [126]:
result = self_consistent_answer(reasoning_samples)
print(result)

VoteResult(winner=18.0, tally={18.0: 3, 26.0: 1, 32.0: 1}, samples=5, confidence=0.6)


In [127]:
def sample_cot_paths(question: str, n_samples: int = 5, temperature: float = 0.7):
     samples = []

     for i in range(n_samples):
         print(f"Generating sample {i + 1}/{n_samples}... provider is: {provider.name}")

         response = llm_reply(
              question,
              temperature=temperature
         )

         samples.append(response)

     return samples

In [128]:
def run_self_consistency(question, n_samples=5, temperature=0.7):
    reasoning_samples = sample_cot_paths(question, n_samples, temperature)

    return self_consistent_answer(reasoning_samples)

In [129]:
result = run_self_consistency(PROMPT)

print(result)

Generating sample 1/5... provider is: Groq


Generating sample 2/5... provider is: Groq
Generating sample 3/5... provider is: Groq
Generating sample 4/5... provider is: Groq
Generating sample 5/5... provider is: Groq
VoteResult(winner=31.0, tally={31.0: 5}, samples=5, confidence=1.0)
